In [ ]:
import os
import shutil

def fast_copy():
    # Hardcoded exact paths discovered in our previous debugging steps
    src = '/kaggle/input/datasets/vaatsalyasrivastava/raf-ravssv0/raf_ravss'
    dst = '/kaggle/working/code'
    
    if not os.path.exists(src):
        print(f"Error: Source path not found at {src}. Did the dataset change?")
        return

    # Clean up destination if it exists
    if os.path.exists(dst):
        print("Clearing old working directory...")
        shutil.rmtree(dst)
        
    # Copy files directly without searching
    print("Copying files...")
    shutil.copytree(src, dst)
    print("Files copied successfully to /kaggle/working/code!")

fast_copy()

In [ ]:
%cd /kaggle/working/code

In [ ]:
!pip install pesq pystoi

In [ ]:
path = "/kaggle/working/code/models/separator.py"   # adjust if your code lives elsewhere

with open(path) as f:
    code = f.read()

old = '''        out = self.mask_head(x)                             # (B, N_SOURCES*N, L)
        out = out.view(B, self.n_sources, N, orig_L)
        masks = F.relu(out)
        return masks'''

new = '''        out = self.mask_head(x)                             # (B, N_SOURCES*N, L)
        out = out.view(B, self.n_sources, N, orig_L)
        # Sigmoid instead of ReLU: bounds the mask to [0,1], which fixes the
        # SI-SDR/SDR amplitude-drift gap. Deviates from Refactor.md Section 5's
        # literal "ReLU" spec -- deliberate tradeoff, see chat for rationale.
        masks = torch.sigmoid(out)
        return masks'''

assert old in code, "pattern not found -- open separator.py and check the mask_head/forward section manually"
code = code.replace(old, new)

with open(path, "w") as f:
    f.write(code)

print("patched separator.py: mask_head now uses sigmoid instead of ReLU")

In [ ]:
# 1. Update EPOCHS from 25 to 18 in config.py
!sed -i 's/EPOCHS\s*=\s*25/EPOCHS            = 18/g' /kaggle/working/code/config.py

# 2. Print the changed line to confirm
print("--- Confirming updated Config: ---")
!grep -i "EPOCHS =" /kaggle/working/code/config.py

# 3. Resume training
print("\n--- Resuming Training with 18 Epochs Limit ---")
%cd /kaggle/working/code
!python train.py

In [ ]:
import os
import zipfile

src_dir = "/kaggle/input/datasets/vaatsalyasrivastava/checkpoints/step_11354"
dst_path = "/kaggle/working/checkpoints/step_11354.pt"

os.makedirs("/kaggle/working/checkpoints", exist_ok=True)

# PyTorch's zip format expects every entry prefixed with one consistent
# top-level archive name (Kaggle stripped this when it auto-extracted the
# .pt on upload). We rebuild it here, using "archive" as that name.
archive_name = "archive"

with zipfile.ZipFile(dst_path, "w", compression=zipfile.ZIP_STORED) as zf:
    for root, dirs, files in os.walk(src_dir):
        for fname in files:
            full_path = os.path.join(root, fname)
            rel_path = os.path.relpath(full_path, src_dir)
            arcname = os.path.join(archive_name, rel_path)
            zf.write(full_path, arcname)

print("Rebuilt checkpoint at:", dst_path)
print("Size:", os.path.getsize(dst_path), "bytes")

In [ ]:
import torch

ckpt = torch.load(dst_path, map_location="cpu", weights_only=False)
print("Loaded successfully. Keys:", list(ckpt.keys()))
print("Step:", ckpt.get("step"), "| Epoch:", ckpt.get("epoch"))

In [ ]:
path = "/kaggle/working/code/metrics_eval.py"

with open(path) as f:
    code = f.read()

old = 'ckpt = torch.load(ckpt_path, map_location=device)'
new = 'ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)'

assert old in code, "pattern not found -- check metrics_eval.py manually, may already be patched"
code = code.replace(old, new)

with open(path, "w") as f:
    f.write(code)

print("patched -- weights_only=False set for checkpoint loading")

In [ ]:
with open("/kaggle/working/code/metrics_eval.py") as f:
    print("weights_only=False" in f.read())

In [ ]:
import glob, os

ckpts = sorted(
    glob.glob("/kaggle/working/checkpoints/step_*.pt"),
    key=lambda p: int(os.path.basename(p).split("_")[1].split(".")[0])
)

if not ckpts:
    print("No checkpoints found in /kaggle/working/checkpoints/ -- nothing to evaluate yet.")
else:
    latest_ckpt = ckpts[-1]
    step_num = os.path.basename(latest_ckpt).split("_")[1].split(".")[0]
    print(f"Latest checkpoint: {latest_ckpt} (step {step_num})")

    # unique output dir per step -> never overwritten by a later/earlier run
    out_dir = f"/kaggle/working/eval_results/step_{step_num}"
    print(f"Writing eval results to: {out_dir}")

    !python metrics_eval.py --ckpt {latest_ckpt} --split val --max_samples 50 --save_audio 5 --out_dir {out_dir}

In [ ]:
import glob
from IPython.display import Audio, display

# reuse the same out_dir computed above -- guarantees you're listening to
# the checkpoint you just evaluated, not some earlier leftover run
audio_dir = f"{out_dir}/audio_samples"
files = sorted(glob.glob(f"{audio_dir}/*.wav"))

if not files:
    print(f"No audio files found in {audio_dir}")
else:
    n_samples = len(set(f.split("/")[-1].split("_")[0] for f in files))
    print(f"Step {step_num} -- found {len(files)} audio files across {n_samples} sample(s)\n")

    for i in range(n_samples):
        print(f"===== sample{i} (checkpoint step {step_num}) =====")
        for kind in ["input_mix", "target_speaker1", "target_speaker2",
                     "estimated_speaker1", "estimated_speaker2"]:
            path = f"{audio_dir}/sample{i}_{kind}.wav"
            if path in files:
                print(kind)
                display(Audio(path))
        print()

# show the numeric summary for this same checkpoint
try:
    with open(f"{out_dir}/summary.csv") as f:
        print(f"===== summary metrics (step {step_num}) =====")
        print(f.read())
except FileNotFoundError:
    print(f"No summary found at {out_dir}/summary.csv")